In [7]:
import pandas as pd
import os
from pathlib import Path

PROCESSED_DIR = Path(os.environ.get("PROCESSED_DIR", "../data/processed"))
BI_EXPORT_DIR = Path(os.environ.get("BI_EXPORT_DIR", "../data/bi_exports"))
BI_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

In [8]:
clean_transactions = pd.read_parquet(PROCESSED_DIR / "clean_transactions.parquet")

clean_transactions.columns = (
    clean_transactions.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

clean_transactions = clean_transactions.rename(columns={
    "invoiceno": "invoice_no",
    "stockcode": "stock_code",
    "invoicedate": "invoice_date",
    "unitprice": "unit_price",
    "customerid": "customer_id"
})
customer_metrics = pd.read_parquet(PROCESSED_DIR / "customer_metrics.parquet")

lifecycle = pd.read_csv(PROCESSED_DIR / "customer_lifecycle_revenue_split.csv")
deciles = pd.read_csv(PROCESSED_DIR / "customer_revenue_deciles.csv")
retention_matrix = pd.read_csv(PROCESSED_DIR / "retention_matrix.csv")

In [9]:
clean_transactions.columns = (
    clean_transactions.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

customer_metrics.columns = (
    customer_metrics.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [ ]:
total_identifiable_revenue = customer_metrics["revenue"].sum()

repeat_order_revenue = lifecycle.loc[
    lifecycle["metric"] == "repeat_order_revenue", "value"
].iloc[0]

repeat_order_revenue_pct = repeat_order_revenue / total_identifiable_revenue

top_10_revenue_share = deciles.loc[
    deciles["customer_decile"] == "Top 10%", "revenue_pct"
].iloc[0]

valid_customers = customer_metrics["customer_id"].nunique()
valid_orders = customer_metrics["total_orders"].sum()

repeat_customers = customer_metrics[customer_metrics["total_orders"] > 1]
second_purchase_conversion = len(repeat_customers) / valid_customers

# You already calculated this before, but recalculate safely
df_valid = clean_transactions[clean_transactions["is_valid_order_line"]].copy()
df_valid["invoice_date"] = pd.to_datetime(df_valid["invoice_date"])

customer_orders = (
    df_valid.groupby(["customer_id", "invoice_no"])
    .agg(
        order_date=("invoice_date", "min"),
        order_revenue=("analytical_revenue", "sum")
    )
    .reset_index()
    .sort_values(["customer_id", "order_date", "invoice_no"])
)

customer_orders["order_rank"] = customer_orders.groupby("customer_id").cumcount() + 1

second_orders = customer_orders[customer_orders["order_rank"] == 2].copy()

first_purchase = (
    customer_orders.groupby("customer_id")["order_date"]
    .min()
    .rename("first_purchase")
    .reset_index()
)

second_orders = second_orders.merge(first_purchase, on="customer_id", how="left")
second_orders["days_to_second_purchase"] = (
    second_orders["order_date"] - second_orders["first_purchase"]
).dt.days

median_days_to_second_purchase = second_orders["days_to_second_purchase"].median()

# identifiable revenue coverage
gross_positive_revenue = clean_transactions.loc[
    clean_transactions["is_positive_purchase"],
    "line_revenue_gross"
].sum()

identifiable_revenue = clean_transactions.loc[
    clean_transactions["is_valid_order_line"],
    "analytical_revenue"
].sum()

identifiable_revenue_coverage = identifiable_revenue / gross_positive_revenue

refund_cancel_revenue = clean_transactions.loc[
    clean_transactions["is_refund_or_return"],
    "line_revenue_gross"
].sum()

refund_rate = abs(refund_cancel_revenue) / gross_positive_revenue

executive_kpis = pd.DataFrame({
    "metric": [
        "analytical_lifecycle_revenue",
        "repeat_order_revenue_pct",
        "second_purchase_conversion_pct",
        "median_days_to_second_purchase",
        "top_10_revenue_share_pct",
        "identifiable_revenue_coverage_pct",
        "refund_cancellation_revenue",
        "refund_rate_pct",
        "valid_customers",
        "valid_orders"
    ],
    "value": [
        identifiable_revenue,
        repeat_order_revenue_pct,
        second_purchase_conversion,
        median_days_to_second_purchase,
        top_10_revenue_share,
        identifiable_revenue_coverage,
        refund_cancel_revenue,
        refund_rate,
        valid_customers,
        valid_orders
    ]
})



executive_kpis.to_csv(BI_EXPORT_DIR / "executive_kpis.csv", index=False)

executive_kpis

,metric,value
0,analytical_lifecycle_revenue,8.911408e+06
1,repeat_order_revenue_pct,7.927939e-01
2,second_purchase_conversion_pct,6.558322e-01
3,median_days_to_second_purchase,5.000000e+01
4,top_10_revenue_share_pct,6.137507e-01
5,identifiable_revenue_coverage_pct,8.354431e-01
6,refund_cancellation_revenue,-8.968125e+05
7,refund_rate_pct,8.407603e-02
8,valid_customers,4.338000e+03
9,valid_orders,1.853200e+04


In [11]:
gross_transaction_revenue = clean_transactions["line_revenue_gross"].sum()

net_transactional_revenue = gross_positive_revenue + refund_cancel_revenue

revenue_reconciliation = pd.DataFrame({
    "revenue_layer": [
        "Gross Positive Purchase Revenue",
        "Refund / Cancellation Revenue",
        "Net Transactional Revenue",
        "Analytical Lifecycle Revenue",
        "Identifiable Customer Revenue"
    ],
    "value": [
        gross_positive_revenue,
        refund_cancel_revenue,
        net_transactional_revenue,
        gross_positive_revenue,
        identifiable_revenue
    ],
    "interpretation": [
        "All positive purchase revenue before refunds/cancellations",
        "Negative revenue from refunds and cancelled invoices",
        "Gross positive revenue after refunds/cancellations",
        "Gross positive revenue used for behavioural lifecycle analysis",
        "Lifecycle revenue linked to identifiable customers"
    ]
})

revenue_reconciliation.to_csv(BI_EXPORT_DIR / "revenue_reconciliation.csv", index=False)

revenue_reconciliation

,revenue_layer,value,interpretation
0,Gross Positive Purchase Revenue,1.066668e+07,All positive purchase revenue before refunds/c...
1,Refund / Cancellation Revenue,-8.968125e+05,Negative revenue from refunds and cancelled in...
2,Net Transactional Revenue,9.769872e+06,Gross positive revenue after refunds/cancellat...
3,Analytical Lifecycle Revenue,1.066668e+07,Gross positive revenue used for behavioural li...
4,Identifiable Customer Revenue,8.911408e+06,Lifecycle revenue linked to identifiable custo...


In [12]:
lifecycle.to_csv(BI_EXPORT_DIR / "lifecycle_revenue_split.csv", index=False)
deciles.to_csv(BI_EXPORT_DIR / "customer_revenue_deciles.csv", index=False)
retention_matrix.to_csv(BI_EXPORT_DIR / "retention_matrix.csv", index=False)

In [13]:
repeat_timing = second_orders[[
    "customer_id",
    "order_date",
    "first_purchase",
    "days_to_second_purchase"
]].copy()

repeat_timing.to_csv(BI_EXPORT_DIR / "repeat_timing.csv", index=False)

repeat_timing.head()

,customer_id,order_date,first_purchase,days_to_second_purchase
0,12347,2011-01-26 14:30:00,2010-12-07 14:57:00,49
1,12348,2011-01-25 10:42:00,2010-12-16 19:09:00,39
2,12352,2011-03-01 14:57:00,2011-02-16 12:33:00,13
3,12356,2011-04-08 12:33:00,2011-01-18 09:50:00,80
4,12358,2011-12-08 10:26:00,2011-07-12 10:04:00,149


In [14]:
list(BI_EXPORT_DIR.glob("*.csv"))

[PosixPath('../data/bi_exports/retention_matrix.csv'),
 PosixPath('../data/bi_exports/customer_revenue_deciles.csv'),
 PosixPath('../data/bi_exports/revenue_reconciliation.csv'),
 PosixPath('../data/bi_exports/repeat_timing.csv'),
 PosixPath('../data/bi_exports/lifecycle_revenue_split.csv'),
 PosixPath('../data/bi_exports/executive_kpis.csv')]

In [15]:
executive_kpis_display = executive_kpis.copy()

def format_metric(row):
    metric = row["metric"]
    value = row["value"]

    if "pct" in metric:
        return f"{value * 100:.1f}%"

    if "revenue" in metric and "pct" not in metric:
        return f"£{value/1_000_000:.2f}M"

    if "days" in metric:
        return f"{int(value)} days"

    if metric in ["valid_customers", "valid_orders"]:
        return f"{int(value):,}"

    return str(value)

executive_kpis_display["display_value"] = executive_kpis_display.apply(
    format_metric,
    axis=1
)

executive_kpis_display

,metric,value,display_value
0,analytical_lifecycle_revenue,8.911408e+06,£8.91M
1,repeat_order_revenue_pct,7.927939e-01,79.3%
2,second_purchase_conversion_pct,6.558322e-01,65.6%
3,median_days_to_second_purchase,5.000000e+01,50 days
4,top_10_revenue_share_pct,6.137507e-01,61.4%
5,identifiable_revenue_coverage_pct,8.354431e-01,83.5%
6,refund_cancellation_revenue,-8.968125e+05,£-0.90M
7,refund_rate_pct,8.407603e-02,8.4%
8,valid_customers,4.338000e+03,"4,338"
9,valid_orders,1.853200e+04,"18,532"


In [16]:
executive_kpis_display.to_csv(
    BI_EXPORT_DIR / "executive_kpis_display.csv",
    index=False
)